In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports and Plot Style ===
import os
from pathlib import Path
import random
from datetime import datetime
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.signal import savgol_filter

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

# --- Plot font: first installed entry of PREFERRED_SANS_FONTS ---
PREFERRED_SANS_FONTS = [
    "DejaVu Sans",]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_selected_font = next((n for n in PREFERRED_SANS_FONTS if n in _available_fonts), "DejaVu Sans")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = PREFERRED_SANS_FONTS
plt.rcParams["mathtext.fontset"] = "dejavusans"
print(f"Plot font in use: {_selected_font}")

# --- CSS for widget section headers and status boxes ---
NOTEBOOK_THEME_CSS = """
<style>
.ir-section-title {
    font-weight: 700;
    color: #1f2937;
    margin: 0 0 6px 0;
    font-size: 16px;
}
.ir-section-subtitle {
    color: #4b5563;
    margin: 0 0 10px 0;
    font-size: 12px;
}
.ir-status {
    padding: 6px 10px;
    border-radius: 6px;
    border: 1px solid #d1d5db;
    background: #f8fafc;
    color: #111827;
    font-size: 12px;
}
</style>
"""

display(HTML(NOTEBOOK_THEME_CSS))


In [ ]:
# === Cell 3: Config and Helpers ===

# --- Dataset paths ---
RAW_DATA_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/raw_spectra")
DATASET_PARENT = RAW_DATA_ROOT.parent
GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/spectra_ground_truth_v7.csv")

# NIR and MIR dataset parents listed in the source dropdowns
NIR_DATASET_PARENT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset")
MIR_DATASET_PARENT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Initial_MIR_Dataset")

# --- Preprocessing defaults ---
DEFAULT_CUT_RANGE = {"lower": 4000.0, "upper": 9000.0}
DEFAULT_ALS = {"lam": 1_000_000.0, "p": 0.01, "niter": 10}
DEFAULT_SG_SMOOTH = {"window_length": 11, "polyorder": 2}
DEFAULT_SG_DERIV = {"window_length": 11, "polyorder": 2, "deriv_order": 1}
DEFAULT_INVERT_X = True
FILENAME_REGEX = re.compile(r"^(?P<material>[A-Za-z0-9]+)_(?P<fixed>obj)_(?P<object_number>\d+)_(?P<region>[A-Za-z0-9]+)$")

# Ground-truth CSV column mapping (region -> GT column)
REGION_TO_GT_COLUMN = {
    "body": "body_class",
    "label": "label_class",
    "cap": "cap_class",
}

# --- Widget layout constants ---
SHARED_LAYOUT = widgets.Layout(width="220px")
WIDE_LAYOUT = widgets.Layout(width="340px")
BUTTON_LAYOUT = widgets.Layout(width="150px", height="34px")
OUTPUT_LAYOUT = widgets.Layout(width="100%", border="1px solid #d1d5db", padding="8px")
BOX_LAYOUT = widgets.Layout(width="100%", border="1px solid #d1d5db", padding="12px", margin="8px 0")
WIDGET_STYLE = {"description_width": "initial"}
BUTTON_STYLE_MAP = {
    "primary": "info",
    "success": "success",
    "warning": "warning",
    "danger": "danger",
    "neutral": "",
}


# --- Widget factory helpers ---
def section_header(title, subtitle=""):
    subtitle_html = f'<div class="ir-section-subtitle">{subtitle}</div>' if subtitle else ""
    return widgets.HTML(
        f'<div class="ir-section-title">{title}</div>{subtitle_html}'
    )


def status_html(message="Ready."):
    return widgets.HTML(f'<div class="ir-status">{message}</div>')


def make_dropdown(description, options, value=None, width="220px"):
    return widgets.Dropdown(
        options=options,
        value=value,
        description=description,
        layout=widgets.Layout(width=width),
        style=WIDGET_STYLE,
    )


def make_float(description, value, step=1.0, width="220px"):
    return widgets.FloatText(
        value=value,
        step=step,
        description=description,
        layout=widgets.Layout(width=width),
        style=WIDGET_STYLE,
    )


def make_int(description, value, width="220px"):
    return widgets.BoundedIntText(
        value=value,
        min=1,
        max=10_000,
        description=description,
        layout=widgets.Layout(width=width),
        style=WIDGET_STYLE,
    )


def make_text(description, value="", placeholder="", width="220px"):
    return widgets.Text(
        value=value,
        placeholder=placeholder,
        description=description,
        layout=widgets.Layout(width=width),
        style=WIDGET_STYLE,
    )


def make_checkbox(description, value=False, width="160px"):
    return widgets.Checkbox(
        value=value,
        description=description,
        layout=widgets.Layout(width=width),
        style=WIDGET_STYLE,
    )


def make_button(description, kind="primary", icon=""):
    return widgets.Button(
        description=description,
        icon=icon,
        button_style=BUTTON_STYLE_MAP.get(kind, ""),
        layout=BUTTON_LAYOUT,
        style=WIDGET_STYLE,
    )


def make_output(height=None):
    layout_kwargs = {
        "width": OUTPUT_LAYOUT.width,
        "border": OUTPUT_LAYOUT.border,
        "padding": OUTPUT_LAYOUT.padding,
    }
    if height is not None:
        layout_kwargs["height"] = height
    return widgets.Output(layout=widgets.Layout(**layout_kwargs))


def wrap_panel(title, subtitle, controls, status_widget, plot_output):
    return widgets.VBox(
        [
            section_header(title, subtitle),
            controls,
            status_widget,
            plot_output,
        ],
        layout=BOX_LAYOUT,
    )


# --- Timestamp and file discovery ---
def now_timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def discover_spectrum_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file()], key=lambda p: p.name)


def discover_preprocessing_folders(parent_dir, prefix):
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        return []
    folders = [p for p in parent_dir.iterdir() if p.is_dir() and p.name.startswith(prefix)]
    return sorted(folders, key=lambda p: p.name, reverse=True)


def find_latest_folder(parent_dir, prefix):
    folders = discover_preprocessing_folders(parent_dir, prefix)
    return folders[0] if folders else None


# --- Filename parsing ---
def validate_filename_pattern(filename):
    return FILENAME_REGEX.match(Path(filename).name) is not None


def parse_filename_metadata(path_or_name):
    name = Path(path_or_name).name
    # Strip any extension that might be present
    stem = Path(name).stem if "." in name else name
    match = FILENAME_REGEX.match(stem)
    if not match:
        return None
    info = match.groupdict()
    return {
        "filename": stem,
        "material": info["material"],
        "fixed_token": info["fixed"],
        "object_number": info["object_number"],
        "region": info["region"],
    }


def build_metadata_table(files):
    valid_rows = []
    invalid_rows = []
    for path in files:
        parsed = parse_filename_metadata(path)
        if parsed is None:
            invalid_rows.append({"filename": path.name, "path": str(path), "error": "Invalid filename pattern"})
        else:
            valid_rows.append({**parsed, "path": str(path)})
    valid_df = pd.DataFrame(valid_rows)
    invalid_df = pd.DataFrame(invalid_rows)
    if not valid_df.empty:
        valid_df = valid_df.sort_values(["material", "object_number", "region", "filename"]).reset_index(drop=True)
    return valid_df, invalid_df


# --- Spectrum I/O ---
def read_spectrum_file(path):
    """Read a two-column spectrum file (wavenumber, signal).

    Skips "##" header lines (raw MIR .txt format) and accepts semicolon-separated
    (preprocessed) or whitespace-separated (raw) data.
    Returns a DataFrame with columns ["wavenumber", "signal"].
    """
    path = Path(path)
    raw_text = path.read_text(encoding="utf-8", errors="replace")
    lines = raw_text.strip().splitlines()
    # Drop "##" header lines (raw MIR .txt format)
    data_lines = [l for l in lines if not l.startswith("##")]
    if not data_lines:
        raise ValueError("File is empty after stripping headers.")
    from io import StringIO
    block = "\n".join(data_lines)
    # Separator: semicolon (preprocessed) or whitespace (raw)
    if ";" in data_lines[0]:
        df = pd.read_csv(StringIO(block), sep=";", header=None, names=["wavenumber", "signal"], engine="python")
    else:
        df = pd.read_csv(StringIO(block), sep=r"\s+", header=None, names=["wavenumber", "signal"], engine="python")
    if df.shape[1] != 2:
        raise ValueError("Expected exactly two columns.")
    df["wavenumber"] = pd.to_numeric(df["wavenumber"], errors="coerce")
    df["signal"] = pd.to_numeric(df["signal"], errors="coerce")
    if df[["wavenumber", "signal"]].isna().any().any():
        raise ValueError("File contains non-numeric values or malformed rows.")
    if df.empty:
        raise ValueError("File is empty.")
    return df.reset_index(drop=True)


def write_spectrum_file(path, df):
    """Write a two-column spectrum file without extension."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = df[["wavenumber", "signal"]].copy()
    out.to_csv(path, sep=";", header=False, index=False, float_format="%.10f")


def is_monotonic_wavenumber(df):
    wn = df["wavenumber"].to_numpy()
    return bool(np.all(np.diff(wn) >= 0) or np.all(np.diff(wn) <= 0))


# --- Ground-truth CSV loader ---
def load_ground_truth(csv_path):
    """Load the ground-truth CSV and return a DataFrame.

    Expected columns: filename, body_class, label_class, cap_class.
    'filename' holds object stems such as 'HDPE_obj_01'.
    """
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f"WARNING: ground-truth CSV not found at {csv_path}")
        return pd.DataFrame()
    gt = pd.read_csv(csv_path, sep=";")
    if "filename" not in gt.columns:
        # Fall back to comma separator
        gt = pd.read_csv(csv_path, sep=",")
    if "filename" not in gt.columns:
        print(f"WARNING: ground-truth CSV has no 'filename' column. Columns: {list(gt.columns)}")
        return pd.DataFrame()
    return gt


def lookup_gt_class(gt_df, object_stem, region):
    """Look up the ground-truth material class for an object + region."""
    if gt_df.empty:
        return "unknown"
    col = REGION_TO_GT_COLUMN.get(region)
    if col is None or col not in gt_df.columns:
        return "unknown"
    row = gt_df[gt_df["filename"] == object_stem]
    if row.empty:
        return "unknown"
    val = row.iloc[0][col]
    if pd.isna(val) or str(val).strip() == "":
        return "unknown"
    return str(val).strip()


# --- Cut, baseline correction, normalization ---
def cut_spectrum(df, lower, upper):
    lo = min(lower, upper)
    hi = max(lower, upper)
    mask = (df["wavenumber"] >= lo) & (df["wavenumber"] <= hi)
    cut_df = df.loc[mask, ["wavenumber", "signal"]].copy().reset_index(drop=True)
    if cut_df.empty:
        raise ValueError(f"Empty cut range for bounds {lo} to {hi} 1/cm.")
    return cut_df


def als_baseline(y, lam=1e7, p=0.01, niter=10, peak_direction="negative", tol=0):
    y = np.asarray(y, dtype=float)

    if y.ndim != 1:
        raise ValueError("Input spectrum must be one-dimensional.")
    if lam <= 0:
        raise ValueError("lam must be > 0.")
    if not (0 < p < 1):
        raise ValueError("p must be between 0 and 1.")
    if niter < 1:
        raise ValueError("niter must be >= 1.")
    if peak_direction not in {"positive", "negative"}:
        raise ValueError("peak_direction must be 'positive' or 'negative'.")

    m = y.size
    if m < 3:
        raise ValueError("Spectrum must contain at least 3 points for ALS baseline correction.")

    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(m - 2, m), format="csc")
    w = np.ones(m, dtype=float)

    for _ in range(int(niter)):
        W = sparse.spdiags(w, 0, m, m)
        Z = W + lam * (D.T @ D)
        z = spsolve(Z, w * y)

        if peak_direction == "positive":
            w_new = p * (y > z) + (1 - p) * (y <= z)
        else:  # negative peaks
            w_new = p * (y < z) + (1 - p) * (y >= z)

        if np.allclose(w, w_new, atol=tol, rtol=0):
            w = w_new
            break
        w = w_new

    return z, w


def baseline_correct_spectrum(
    df,
    lam,
    p,
    niter,
    peak_direction="negative",
    zero_anchor_quantile=0.995,
    only_shift_down=True,
):
    baseline, weights = als_baseline(
        df["signal"].to_numpy(),
        lam=lam,
        p=p,
        niter=niter,
        peak_direction=peak_direction,
    )

    corrected = df.copy()
    corrected["signal"] = corrected["signal"].to_numpy() - baseline

    anchor_value = float(np.quantile(corrected["signal"], zero_anchor_quantile))

    if only_shift_down:
        anchor_shift = max(anchor_value, 0.0)
    else:
        anchor_shift = anchor_value

    corrected["signal"] = corrected["signal"] - anchor_shift

    baseline_df = df[["wavenumber"]].copy()
    baseline_df["signal"] = baseline

    diagnostics = {
        "anchor_shift": anchor_shift,
        "q50": float(np.quantile(corrected["signal"], 0.50)),
        "q95": float(np.quantile(corrected["signal"], 0.95)),
        "q99": float(np.quantile(corrected["signal"], 0.99)),
        "fraction_above_zero": float(np.mean(corrected["signal"].to_numpy() > 0)),
        "peak_direction": peak_direction,
    }

    return corrected, baseline_df, weights, diagnostics


def zscore_normalize_spectrum(df):
    values = df["signal"].to_numpy(dtype=float)
    mean = float(values.mean())
    std = float(values.std(ddof=0))
    notes = []
    if np.isclose(std, 0.0):
        normalized_values = np.zeros_like(values)
        notes.append("zero_std_detected")
    else:
        normalized_values = (values - mean) / std
    normalized = df.copy()
    normalized["signal"] = normalized_values
    return normalized, {"mean": mean, "std": std, "notes": notes}


# --- Savitzky-Golay smoothing and derivative ---
def smooth_spectrum(df, window_length, polyorder):
    """Savitzky-Golay smoothing of a spectrum.

    Fits a low-order polynomial in a sliding window, suppressing high-frequency
    noise while largely preserving peak position, height and width.

    Parameters
    ----------
    window_length : int
        Length of the SG filter window. Must be odd and greater than polyorder.
    polyorder : int
        Order of the polynomial used to fit the samples.
    """
    y = df["signal"].to_numpy(dtype=float)
    n = y.size
    wl = int(window_length)
    po = int(polyorder)

    if wl % 2 == 0:
        raise ValueError(f"window_length must be odd (got {wl}).")
    if wl <= po:
        raise ValueError(f"window_length ({wl}) must be greater than polyorder ({po}).")
    if wl > n:
        raise ValueError(f"window_length ({wl}) exceeds spectrum length ({n}).")
    if po < 0:
        raise ValueError("polyorder must be >= 0.")

    smoothed_values = savgol_filter(y, window_length=wl, polyorder=po, deriv=0)
    smoothed = df.copy()
    smoothed["signal"] = smoothed_values
    return smoothed, {"window_length": wl, "polyorder": po}


def derivative_spectrum(df, window_length, polyorder, deriv_order):
    """Savitzky-Golay derivative of a spectrum.

    The 1st derivative removes additive offsets; the 2nd derivative also removes
    slope effects. Smoothing is applied jointly by the polynomial fit.

    delta is the median absolute wavenumber step, so the derivative is per 1/cm.
    """
    x = df["wavenumber"].to_numpy(dtype=float)
    y = df["signal"].to_numpy(dtype=float)
    n = y.size
    wl = int(window_length)
    po = int(polyorder)
    do = int(deriv_order)

    if wl % 2 == 0:
        raise ValueError(f"window_length must be odd (got {wl}).")
    if wl <= po:
        raise ValueError(f"window_length ({wl}) must be greater than polyorder ({po}).")
    if wl > n:
        raise ValueError(f"window_length ({wl}) exceeds spectrum length ({n}).")
    if do < 1:
        raise ValueError("deriv_order must be >= 1.")
    if po < do:
        raise ValueError(f"polyorder ({po}) must be >= deriv_order ({do}).")

    steps = np.abs(np.diff(x))
    delta = float(np.median(steps)) if steps.size and np.median(steps) > 0 else 1.0

    deriv_values = savgol_filter(y, window_length=wl, polyorder=po, deriv=do, delta=delta)
    derived = df.copy()
    derived["signal"] = deriv_values
    return derived, {"window_length": wl, "polyorder": po, "deriv_order": do, "delta": delta}


# --- Graph styling and PNG export ---
# Thesis colour scheme (RGB 0-255 scaled to 0-1)
COLOR_INPUT = (138 / 255, 190 / 255, 231 / 255)        # light blue  - input spectra
COLOR_OUTPUT = (151 / 255, 41 / 255, 130 / 255)        # magenta     - processed output
COLOR_INTERMEDIATE = (90 / 255, 86 / 255, 161 / 255)   # violet      - intermediate (e.g. fitted baseline)

# Line style per role, shared by all preview plots
INPUT_STYLE = {"color": COLOR_INPUT, "alpha": 0.7, "linewidth": 1.8}
INTERMEDIATE_STYLE = {"color": COLOR_INTERMEDIATE, "alpha": 1.0, "linewidth": 2.0}
OUTPUT_STYLE = {"color": COLOR_OUTPUT, "alpha": 1.0, "linewidth": 2.2}

# Output folder for exported PNG figures (created on first save)
GRAPH_EXPORT_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/03_IR_Preprocessing")
GRAPH_EXPORT_DPI = 300


class FigureHolder:
    """Most recently rendered figure of a panel, exported by its Save PNG button.

    ``descriptor`` is appended to the export filename.
    """

    def __init__(self, slug):
        self.slug = slug
        self.fig = None
        self.descriptor = ""

    def set(self, fig, descriptor=""):
        self.fig = fig
        self.descriptor = descriptor


def make_save_button():
    return make_button("Save PNG", kind="neutral", icon="download")


def save_panel_figure(holder, status_widget):
    """Save the panel's current figure as a PNG into GRAPH_EXPORT_DIR."""
    if holder.fig is None:
        status_widget.value = (
            '<div class="ir-status">No figure to save yet - adjust the controls '
            'to render a preview first.</div>'
        )
        return
    try:
        GRAPH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        descriptor = re.sub(r"[^A-Za-z0-9._-]+", "_", holder.descriptor).strip("_")
        stem = f"{holder.slug}_{descriptor}" if descriptor else holder.slug
        out_path = GRAPH_EXPORT_DIR / f"{now_timestamp()}_{stem}.png"
        holder.fig.savefig(out_path, dpi=GRAPH_EXPORT_DPI, bbox_inches="tight", facecolor="white")
        status_widget.value = f'<div class="ir-status">Saved PNG -> {out_path}</div>'
    except Exception as exc:
        status_widget.value = f'<div class="ir-status">PNG export failed: {exc}</div>'


# --- Plotting helpers ---
def apply_inverse_xaxis(ax, enabled=True):
    if enabled:
        ax.invert_xaxis()
    return ax


def plot_single_spectra(ax, spectra_items, title="Spectrum preview", xlabel="Wavenumber (1/cm)", ylabel="Signal", invert_x=True):
    plotted = False
    for item in spectra_items:
        if item is None:
            continue
        ax.plot(item["x"], item["y"], label=item["label"], color=item.get("color", None), linewidth=item.get("linewidth", 1.8), alpha=item.get("alpha", 1.0))
        plotted = True
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if plotted:
        ax.legend(loc="best")
    apply_inverse_xaxis(ax, enabled=invert_x)
    return ax


def choose_random_file(file_options):
    if not file_options:
        return None
    return random.choice(list(file_options))


# --- Batch export ---
def write_parameter_log(log_path, lines):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text("".join(str(line) for line in lines), encoding="utf-8")


def export_batch_results(step_name, source_files, processor_fn, output_prefix, parent_dir, parameter_lines):
    timestamp = now_timestamp()
    parent_dir = Path(parent_dir)
    output_dir = parent_dir / f"{output_prefix}_{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)

    records = []
    failures = []
    for source_path in source_files:
        source_path = Path(source_path)
        try:
            processed_df, details = processor_fn(source_path)
            write_spectrum_file(output_dir / source_path.stem, processed_df)
            record = {
                "filename": source_path.stem,
                "status": "success",
                "output_path": str(output_dir / source_path.stem),
            }
            if isinstance(details, dict):
                record.update(details)
            records.append(record)
        except Exception as exc:
            failures.append({"filename": source_path.stem, "error": str(exc)})
            records.append({"filename": source_path.stem, "status": "failed", "output_path": "", "error": str(exc)})

    summary_df = pd.DataFrame(records)
    log_path = parent_dir / f"{output_prefix}_params_{timestamp}.txt"
    lines = list(parameter_lines)
    lines.extend([
        f"output_folder: {output_dir}",
        f"number_of_files_attempted: {len(source_files)}",
        f"number_of_files_processed_successfully: {int((summary_df['status'] == 'success').sum()) if not summary_df.empty else 0}",
        f"number_of_failed_files: {len(failures)}",
    ])
    if failures:
        lines.append("failed_files:")
        for item in failures:
            lines.append(f"- {item['filename']}: {item['error']}")
    else:
        lines.append("failed_files: none")
    write_parameter_log(log_path, lines)
    return output_dir, log_path, summary_df, failures


def folder_metadata_table(folder):
    files = discover_spectrum_files(folder)
    return build_metadata_table(files)


def make_folder_dropdown_options(folders, allow_none=True):
    options = []
    if allow_none:
        options.append(("<None>", None))
    options.extend([(p.name, str(p)) for p in folders])
    return options


def update_dropdown_from_files(dropdown_widget, files, keep_value=True):
    names = [p.name for p in files]
    dropdown_widget.options = names
    if not names:
        dropdown_widget.value = None
        return
    if keep_value and dropdown_widget.value in names:
        return
    dropdown_widget.value = names[0]


def get_file_map(folder):
    files = discover_spectrum_files(folder)
    return {p.name: p for p in files}


# --- Load ground truth and run sanity checks ---
GT_DF = load_ground_truth(GROUND_TRUTH_CSV)

raw_files_all_entries = list(RAW_DATA_ROOT.iterdir()) if RAW_DATA_ROOT.exists() else []
raw_files = discover_spectrum_files(RAW_DATA_ROOT)
raw_metadata_df, invalid_filename_df = build_metadata_table(raw_files)
duplicate_filenames = raw_metadata_df[raw_metadata_df.duplicated(subset=["filename"], keep=False)] if not raw_metadata_df.empty else pd.DataFrame()

print("=== IR preprocessing configuration ===")
print(f"raw data root: {RAW_DATA_ROOT}")
print(f"dataset parent path: {DATASET_PARENT}")
print(f"ground truth CSV: {GROUND_TRUTH_CSV} (exists={GROUND_TRUTH_CSV.exists()})")
print(f"NIR dataset parent: {NIR_DATASET_PARENT}")
print(f"MIR dataset parent: {MIR_DATASET_PARENT}")
print(f"default cut range: {DEFAULT_CUT_RANGE}")
print(f"default ALS parameters: {DEFAULT_ALS}")
print("pipeline order: raw -> [optional cut] -> baseline corrected -> normalized")
print()
print("=== Sanity checks ===")
print(f"number of filesystem entries found: {len(raw_files_all_entries)}")
print(f"number of valid files found: {len(raw_metadata_df)}")
print(f"number of invalid filename patterns: {len(invalid_filename_df)}")
print(f"warning if duplicate filenames exist: {not duplicate_filenames.empty}")
if not GT_DF.empty:
    print(f"ground truth rows: {len(GT_DF)}, columns: {list(GT_DF.columns)}")

if not raw_metadata_df.empty:
    print("Parsed metadata preview:")
    display(raw_metadata_df.head(10))
else:
    print("Parsed metadata preview: no valid files found.")

if not invalid_filename_df.empty:
    print("Invalid filename preview:")
    display(invalid_filename_df.head(10))

if not duplicate_filenames.empty:
    print("Duplicate filename warning:")
    display(duplicate_filenames[["filename", "path"]])

if raw_files:
    example_path = raw_files[0]
    print(f"Example file read: {example_path.name}")
    try:
        example_df = read_spectrum_file(example_path)
        display(example_df.head())
        print("two numeric columns verified: True")
        print(f"wavenumber monotonic: {is_monotonic_wavenumber(example_df)}")
        if not is_monotonic_wavenumber(example_df):
            print("warning: wavenumber is not monotonic in the example file.")
    except Exception as exc:
        print(f"example file read failed: {exc}")
else:
    print("No raw files discovered yet. Update RAW_DATA_ROOT if needed and rerun this cell.")


In [ ]:
# === Cell 4: Cut Range (Optional) ===
# Optional step; the Baseline Correction cell can read raw_spectra directly.
# Any subfolder of the NIR or MIR dataset can be selected as source.

def _build_generic_source_options():
    """List all subfolders of both NIR and MIR dataset parents."""
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted([p for p in parent.iterdir() if p.is_dir()], key=lambda p: p.name)
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options

cut_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_generic_source_options(),
    value=_build_generic_source_options()[0][1] if _build_generic_source_options() else None,
    width="500px",
)
cut_file_dropdown = make_dropdown(
    description="Preview file",
    options=[],
    value=None,
    width="340px",
)
cut_lower_widget = make_float("Lower wavenumber", DEFAULT_CUT_RANGE["lower"], step=10.0)
cut_upper_widget = make_float("Upper wavenumber", DEFAULT_CUT_RANGE["upper"], step=10.0)
cut_random_button = make_button("Random", kind="neutral", icon="random")
cut_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
cut_apply_button = make_button("Apply to all files", kind="success", icon="play")
cut_status = status_html("Select a source folder, then preview and export cut spectra.")
cut_plot_output = make_output(height="430px")
cut_batch_output = make_output(height="220px")
cut_save_button = make_save_button()
cut_figure_holder = FigureHolder("cut")


def refresh_cut_files(_=None):
    folder = cut_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(cut_file_dropdown, files, keep_value=False)
    preview_cut_spectrum()


def preview_cut_spectrum(_=None):
    cut_plot_output.clear_output(wait=True)
    folder = cut_source_dropdown.value
    filename = cut_file_dropdown.value

    if folder is None:
        cut_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    if filename is None:
        cut_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    lower = cut_lower_widget.value
    upper = cut_upper_widget.value

    if lower == upper:
        cut_status.value = '<div class="ir-status">Warning: lower and upper bounds are equal — empty or single-point range.</div>'

    try:
        path = Path(folder) / filename
        raw_df = read_spectrum_file(path)
        cut_df = cut_spectrum(raw_df, lower=lower, upper=upper)

        with cut_plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4.8))
            plot_single_spectra(
                ax,
                [
                    {"x": raw_df["wavenumber"], "y": raw_df["signal"], "label": "Source spectrum", **INPUT_STYLE},
                    {"x": cut_df["wavenumber"], "y": cut_df["signal"], "label": f"Cut [{lower:.0f}, {upper:.0f}] 1/cm", **OUTPUT_STYLE},
                ],
                title=f"Cut range: {filename}",
                ylabel="Signal",
                invert_x=DEFAULT_INVERT_X,
            )
            cut_figure_holder.set(fig, filename)
            plt.show()

        cut_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b>. '
            f'Cut range: [{lower:.0f}, {upper:.0f}] 1/cm, '
            f'source points: {len(raw_df)}, cut points: {len(cut_df)}.</div>'
        )

    except Exception as exc:
        cut_status.value = f'<div class="ir-status">Cut preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_cut_file(_):
    if not cut_file_dropdown.options:
        return
    cut_file_dropdown.value = choose_random_file(cut_file_dropdown.options)


def save_cut_figure(_):
    save_panel_figure(cut_figure_holder, cut_status)


def refresh_cut_folder_list(_=None):
    cut_source_dropdown.options = _build_generic_source_options()
    refresh_cut_files()


def apply_cut_to_all(_):
    cut_batch_output.clear_output(wait=True)
    folder = cut_source_dropdown.value

    if folder is None:
        cut_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        cut_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    lower = cut_lower_widget.value
    upper = cut_upper_widget.value
    selected_lower = min(lower, upper)
    selected_upper = max(lower, upper)
    output_parent = Path(folder).parent  # export beside the source folder

    def processor_fn(source_path):
        df = read_spectrum_file(source_path)
        cut_df = cut_spectrum(df, lower=selected_lower, upper=selected_upper)
        return cut_df, {"status": "success", "n_points": len(cut_df)}

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Cut Range",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix="cut_spectra",
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: cut range",
                f"timestamp: {datetime.now().isoformat()}",
                f"source_folder: {folder}",
                f"selected_cut_range: {selected_lower} to {selected_upper}",
            ],
        )

        success_count = int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0
        cut_status.value = (
            f'<div class="ir-status">Cut batch export completed. '
            f'Successes: {success_count}, failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with cut_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        cut_status.value = f'<div class="ir-status">Cut batch export failed: {exc}</div>'


cut_source_dropdown.observe(refresh_cut_files, names="value")
cut_file_dropdown.observe(preview_cut_spectrum, names="value")
cut_lower_widget.observe(preview_cut_spectrum, names="value")
cut_upper_widget.observe(preview_cut_spectrum, names="value")
cut_random_button.on_click(choose_random_cut_file)
cut_refresh_button.on_click(refresh_cut_folder_list)
cut_apply_button.on_click(apply_cut_to_all)
cut_save_button.on_click(save_cut_figure)

cut_controls = widgets.VBox([
    widgets.HBox([cut_source_dropdown, cut_refresh_button]),
    widgets.HBox([cut_file_dropdown, cut_random_button]),
    widgets.HBox([cut_lower_widget, cut_upper_widget, cut_apply_button, cut_save_button]),
    cut_batch_output,
])

cut_panel = wrap_panel(
    title="Cut Range Preview and Export (Optional)",
    subtitle="Select any NIR or MIR subfolder as source, adjust the cut window, and export all cut spectra. Outputs are written beside the selected source folder. This step is optional — Cell 5 can read directly from raw spectra.",
    controls=cut_controls,
    status_widget=cut_status,
    plot_output=cut_plot_output,
)

display(cut_panel)
refresh_cut_files()


In [ ]:
# === Cell 5: Baseline Correction ===
# The mode (Reflectance NIR / Absorbance MIR) sets ALS peak direction and anchoring.
# The source dropdown lists all subfolders of the NIR and MIR datasets.

BC_MODE_PARAMS = {
    "Reflectance (NIR)": {
        "peak_direction": "negative",
        "zero_anchor_quantile": 0.995,
        "only_shift_down": True,
        "ylabel": "Reflectance",
    },
    "Absorbance (MIR)": {
        "peak_direction": "positive",
        "zero_anchor_quantile": 0.005,
        "only_shift_down": False,
        "ylabel": "Absorbance",
    },
}

bc_mode_toggle = widgets.ToggleButtons(
    options=list(BC_MODE_PARAMS.keys()),
    value="Reflectance (NIR)",
    description="Mode",
    style=WIDGET_STYLE,
)

def _build_bc_source_options():
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted([p for p in parent.iterdir() if p.is_dir()], key=lambda p: p.name)
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options

bc_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_bc_source_options(),
    value=_build_bc_source_options()[0][1] if _build_bc_source_options() else None,
    width="500px",
)
bc_file_dropdown = make_dropdown(description="Preview file", options=[], value=None, width="340px")
bc_random_button = make_button("Random", kind="neutral", icon="random")
bc_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
bc_lam_widget = make_float("ALS lam", DEFAULT_ALS["lam"], step=10000.0)
bc_p_widget = make_float("ALS p", DEFAULT_ALS["p"], step=0.001)
bc_niter_widget = make_int("ALS niter", DEFAULT_ALS["niter"])
bc_apply_button = make_button("Apply to all files", kind="success", icon="play")
bc_status = status_html("Select a source folder and mode to preview ALS baseline correction.")
bc_plot_output = make_output(height="430px")
bc_batch_output = make_output(height="220px")
bc_save_button = make_save_button()
bc_figure_holder = FigureHolder("baseline")


def refresh_bc_files(_=None):
    folder = bc_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(bc_file_dropdown, files, keep_value=False)
    preview_baseline()


def preview_baseline(_=None):
    bc_plot_output.clear_output(wait=True)
    folder = bc_source_dropdown.value
    filename = bc_file_dropdown.value

    if folder is None:
        bc_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    if filename is None:
        bc_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    mode_params = BC_MODE_PARAMS[bc_mode_toggle.value]
    lam = bc_lam_widget.value
    p = bc_p_widget.value
    niter = bc_niter_widget.value
    path = Path(folder) / filename

    try:
        input_df = read_spectrum_file(path)
        corrected_df, baseline_df, weights, diagnostics = baseline_correct_spectrum(
            input_df,
            lam=lam,
            p=p,
            niter=niter,
            peak_direction=mode_params["peak_direction"],
            zero_anchor_quantile=mode_params["zero_anchor_quantile"],
            only_shift_down=mode_params["only_shift_down"],
        )

        with bc_plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4.8))
            plot_single_spectra(
                ax,
                [
                    {"x": input_df["wavenumber"], "y": input_df["signal"], "label": "Input spectrum", **INPUT_STYLE},
                    {"x": baseline_df["wavenumber"], "y": baseline_df["signal"], "label": "Estimated baseline", **INTERMEDIATE_STYLE},
                    {"x": corrected_df["wavenumber"], "y": corrected_df["signal"], "label": "Baseline-corrected", **OUTPUT_STYLE},
                ],
                title=f"ALS baseline: {filename}",
                ylabel="Signal",
                invert_x=DEFAULT_INVERT_X,
            )
            bc_figure_holder.set(fig, filename)
            plt.show()

        bc_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b> [{bc_mode_toggle.value}] using '
            f'lam={lam}, p={p}, niter={niter}. '
            f'Anchor shift={diagnostics["anchor_shift"]:.6f}, '
            f'fraction above zero={diagnostics["fraction_above_zero"]:.4f}.</div>'
        )

    except Exception as exc:
        bc_status.value = f'<div class="ir-status">Baseline preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_bc_file(_):
    if not bc_file_dropdown.options:
        return
    bc_file_dropdown.value = choose_random_file(bc_file_dropdown.options)


def save_bc_figure(_):
    save_panel_figure(bc_figure_holder, bc_status)


def refresh_bc_folder_list(_=None):
    bc_source_dropdown.options = _build_bc_source_options()
    refresh_bc_files()


def apply_baseline_to_all(_):
    bc_batch_output.clear_output(wait=True)
    folder = bc_source_dropdown.value

    if folder is None:
        bc_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        bc_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    mode_params = BC_MODE_PARAMS[bc_mode_toggle.value]
    lam = bc_lam_widget.value
    p = bc_p_widget.value
    niter = bc_niter_widget.value
    output_parent = Path(folder).parent  # export beside the source folder

    def processor_fn(source_path):
        input_df = read_spectrum_file(source_path)
        corrected_df, _, _, diagnostics = baseline_correct_spectrum(
            input_df,
            lam=lam,
            p=p,
            niter=niter,
            peak_direction=mode_params["peak_direction"],
            zero_anchor_quantile=mode_params["zero_anchor_quantile"],
            only_shift_down=mode_params["only_shift_down"],
        )
        return corrected_df, {
            "status": "success",
            "n_points": len(corrected_df),
            "anchor_shift": diagnostics["anchor_shift"],
            "fraction_above_zero": diagnostics["fraction_above_zero"],
            "peak_direction": diagnostics["peak_direction"],
        }

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Baseline Correction",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix="bc_spectra",
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: baseline correction",
                f"timestamp: {datetime.now().isoformat()}",
                f"mode: {bc_mode_toggle.value}",
                f"source_folder: {folder}",
                f"als_lam: {lam}",
                f"als_p: {p}",
                f"als_niter: {niter}",
                f"peak_direction: {mode_params['peak_direction']}",
                f"zero_anchor_quantile: {mode_params['zero_anchor_quantile']}",
                f"only_shift_down: {mode_params['only_shift_down']}",
                f"total_attempted: {len(source_files)}",
            ],
        )

        bc_status.value = (
            f'<div class="ir-status">Baseline batch export completed [{bc_mode_toggle.value}]. '
            f'Successes: {int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0}, '
            f'failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with bc_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        bc_status.value = f'<div class="ir-status">Baseline batch export failed: {exc}</div>'


bc_mode_toggle.observe(preview_baseline, names="value")
bc_source_dropdown.observe(refresh_bc_files, names="value")
bc_file_dropdown.observe(preview_baseline, names="value")
bc_lam_widget.observe(preview_baseline, names="value")
bc_p_widget.observe(preview_baseline, names="value")
bc_niter_widget.observe(preview_baseline, names="value")
bc_random_button.on_click(choose_random_bc_file)
bc_refresh_button.on_click(refresh_bc_folder_list)
bc_apply_button.on_click(apply_baseline_to_all)
bc_save_button.on_click(save_bc_figure)

bc_controls = widgets.VBox([
    widgets.HBox([bc_mode_toggle]),
    widgets.HBox([bc_source_dropdown, bc_refresh_button]),
    widgets.HBox([bc_file_dropdown, bc_random_button]),
    widgets.HBox([bc_lam_widget, bc_p_widget, bc_niter_widget, bc_apply_button, bc_save_button]),
    bc_batch_output,
])

bc_panel = wrap_panel(
    title="Baseline Correction Preview and Export",
    subtitle="Select Reflectance (NIR) or Absorbance (MIR) mode, pick any NIR or MIR subfolder as source, and export all baseline-corrected files. Outputs are written beside the selected source folder.",
    controls=bc_controls,
    status_widget=bc_status,
    plot_output=bc_plot_output,
)

display(bc_panel)
refresh_bc_files()


In [ ]:
# === Cell 6: Normalization ===
# Z-score normalization; the source dropdown lists all NIR and MIR subfolders.

def _build_norm_source_options():
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted([p for p in parent.iterdir() if p.is_dir()], key=lambda p: p.name)
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options

norm_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_norm_source_options(),
    value=_build_norm_source_options()[0][1] if _build_norm_source_options() else None,
    width="500px",
)
norm_file_dropdown = make_dropdown(description="Preview file", options=[], value=None, width="340px")
norm_random_button = make_button("Random", kind="neutral", icon="random")
norm_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
norm_apply_button = make_button("Apply to all files", kind="success", icon="play")
norm_status = status_html("Select a source folder to preview z-score normalization.")
norm_plot_output = make_output(height="430px")
norm_batch_output = make_output(height="220px")
norm_save_button = make_save_button()
norm_figure_holder = FigureHolder("normalization")


def refresh_norm_files(_=None):
    folder = norm_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(norm_file_dropdown, files, keep_value=False)
    preview_normalization()


def preview_normalization(_=None):
    norm_plot_output.clear_output(wait=True)
    folder = norm_source_dropdown.value
    filename = norm_file_dropdown.value

    if folder is None:
        norm_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    if filename is None:
        norm_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    path = Path(folder) / filename

    try:
        bc_df = read_spectrum_file(path)
        norm_df, stats = zscore_normalize_spectrum(bc_df)

        with norm_plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4.8))
            plot_single_spectra(
                ax,
                [
                    {"x": bc_df["wavenumber"], "y": bc_df["signal"], "label": "Input spectrum", **INPUT_STYLE},
                    {"x": norm_df["wavenumber"], "y": norm_df["signal"], "label": "Normalized", **OUTPUT_STYLE},
                ],
                title=f"Normalization: {filename}",
                ylabel="Signal",
                invert_x=DEFAULT_INVERT_X,
            )
            norm_figure_holder.set(fig, filename)
            plt.show()

        norm_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b>. '
            f'mean={stats["mean"]:.6f}, std={stats["std"]:.6f}. '
            f'{", ".join(stats["notes"]) if stats["notes"] else "No warnings."}</div>'
        )

    except Exception as exc:
        norm_status.value = f'<div class="ir-status">Normalization preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_norm_file(_):
    if not norm_file_dropdown.options:
        return
    norm_file_dropdown.value = choose_random_file(norm_file_dropdown.options)


def save_norm_figure(_):
    save_panel_figure(norm_figure_holder, norm_status)


def refresh_norm_folder_list(_=None):
    norm_source_dropdown.options = _build_norm_source_options()
    refresh_norm_files()


def apply_normalization_to_all(_):
    norm_batch_output.clear_output(wait=True)
    folder = norm_source_dropdown.value

    if folder is None:
        norm_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        norm_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    output_parent = Path(folder).parent  # export beside the source folder

    def processor_fn(source_path):
        bc_df = read_spectrum_file(source_path)
        norm_df, stats = zscore_normalize_spectrum(bc_df)
        return norm_df, {
            "status": "success",
            "n_points": len(norm_df),
            "std": stats["std"],
            "notes": ",".join(stats["notes"]) if stats["notes"] else "",
        }

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Normalization",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix="norm_spectra",
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: normalization",
                f"timestamp: {datetime.now().isoformat()}",
                f"source_folder: {folder}",
                "normalization_method: z-score",
                f"total_attempted: {len(source_files)}",
            ],
        )

        success_count = int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0
        norm_status.value = (
            f'<div class="ir-status">Normalization batch export completed. '
            f'Successes: {success_count}, failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with norm_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        norm_status.value = f'<div class="ir-status">Normalization batch export failed: {exc}</div>'


norm_source_dropdown.observe(refresh_norm_files, names="value")
norm_file_dropdown.observe(preview_normalization, names="value")
norm_random_button.on_click(choose_random_norm_file)
norm_refresh_button.on_click(refresh_norm_folder_list)
norm_apply_button.on_click(apply_normalization_to_all)
norm_save_button.on_click(save_norm_figure)

norm_controls = widgets.VBox([
    widgets.HBox([norm_source_dropdown, norm_refresh_button]),
    widgets.HBox([norm_file_dropdown, norm_random_button, norm_apply_button, norm_save_button]),
    norm_batch_output,
])

norm_panel = wrap_panel(
    title="Normalization Preview and Export",
    subtitle="Select any NIR or MIR subfolder as source and apply z-score normalization. Outputs are written beside the selected source folder.",
    controls=norm_controls,
    status_widget=norm_status,
    plot_output=norm_plot_output,
)

display(norm_panel)
refresh_norm_files()


In [ ]:
# === Cell 7: Savitzky-Golay Smoothing ===

def _build_smooth_source_options():
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted([p for p in parent.iterdir() if p.is_dir()], key=lambda p: p.name)
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options

smooth_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_smooth_source_options(),
    value=_build_smooth_source_options()[0][1] if _build_smooth_source_options() else None,
    width="500px",
)
smooth_file_dropdown = make_dropdown(description="Preview file", options=[], value=None, width="340px")
smooth_window_widget = make_int("SG window_length", DEFAULT_SG_SMOOTH["window_length"])
smooth_poly_widget = make_int("SG polyorder", DEFAULT_SG_SMOOTH["polyorder"])
smooth_random_button = make_button("Random", kind="neutral", icon="random")
smooth_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
smooth_apply_button = make_button("Apply to all files", kind="success", icon="play")
smooth_status = status_html("Select a source folder to preview Savitzky-Golay smoothing.")
smooth_plot_output = make_output(height="430px")
smooth_batch_output = make_output(height="220px")
smooth_save_button = make_save_button()
smooth_figure_holder = FigureHolder("smoothing")


def refresh_smooth_files(_=None):
    folder = smooth_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(smooth_file_dropdown, files, keep_value=False)
    preview_smoothing()


def preview_smoothing(_=None):
    smooth_plot_output.clear_output(wait=True)
    folder = smooth_source_dropdown.value
    filename = smooth_file_dropdown.value

    if folder is None:
        smooth_status.value = '<div class="ir-status">No source folder selected.</div>'
        return
    if filename is None:
        smooth_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    path = Path(folder) / filename
    try:
        input_df = read_spectrum_file(path)
        smoothed_df, params = smooth_spectrum(
            input_df,
            window_length=smooth_window_widget.value,
            polyorder=smooth_poly_widget.value,
        )

        with smooth_plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4.8))
            plot_single_spectra(
                ax,
                [
                    {"x": input_df["wavenumber"], "y": input_df["signal"], "label": "Input spectrum", **INPUT_STYLE},
                    {"x": smoothed_df["wavenumber"], "y": smoothed_df["signal"], "label": "Smoothed (SG)", **OUTPUT_STYLE},
                ],
                title=f"Savitzky-Golay smoothing: {filename}",
                ylabel="Signal",
                invert_x=DEFAULT_INVERT_X,
            )
            smooth_figure_holder.set(fig, filename)
            plt.show()

        smooth_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b>. '
            f'window_length={params["window_length"]}, polyorder={params["polyorder"]}.</div>'
        )
    except Exception as exc:
        smooth_status.value = f'<div class="ir-status">Smoothing preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_smooth_file(_):
    if not smooth_file_dropdown.options:
        return
    smooth_file_dropdown.value = choose_random_file(smooth_file_dropdown.options)


def save_smooth_figure(_):
    save_panel_figure(smooth_figure_holder, smooth_status)


def refresh_smooth_folder_list(_=None):
    smooth_source_dropdown.options = _build_smooth_source_options()
    refresh_smooth_files()


def apply_smoothing_to_all(_):
    smooth_batch_output.clear_output(wait=True)
    folder = smooth_source_dropdown.value
    if folder is None:
        smooth_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        smooth_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    window_length = smooth_window_widget.value
    polyorder = smooth_poly_widget.value
    output_parent = Path(folder).parent  # export beside the source folder

    def processor_fn(source_path):
        input_df = read_spectrum_file(source_path)
        smoothed_df, params = smooth_spectrum(input_df, window_length=window_length, polyorder=polyorder)
        return smoothed_df, {"status": "success", "n_points": len(smoothed_df), **params}

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Smoothing",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix="smoothed_spectra",
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: smoothing",
                f"timestamp: {datetime.now().isoformat()}",
                f"source_folder: {folder}",
                "method: savitzky-golay",
                f"window_length: {window_length}",
                f"polyorder: {polyorder}",
                f"total_attempted: {len(source_files)}",
            ],
        )

        success_count = int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0
        smooth_status.value = (
            f'<div class="ir-status">Smoothing batch export completed. '
            f'Successes: {success_count}, failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with smooth_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        smooth_status.value = f'<div class="ir-status">Smoothing batch export failed: {exc}</div>'


smooth_source_dropdown.observe(refresh_smooth_files, names="value")
smooth_file_dropdown.observe(preview_smoothing, names="value")
smooth_window_widget.observe(preview_smoothing, names="value")
smooth_poly_widget.observe(preview_smoothing, names="value")
smooth_random_button.on_click(choose_random_smooth_file)
smooth_refresh_button.on_click(refresh_smooth_folder_list)
smooth_apply_button.on_click(apply_smoothing_to_all)
smooth_save_button.on_click(save_smooth_figure)

smooth_controls = widgets.VBox([
    widgets.HBox([smooth_source_dropdown, smooth_refresh_button]),
    widgets.HBox([smooth_file_dropdown, smooth_random_button]),
    widgets.HBox([smooth_window_widget, smooth_poly_widget, smooth_apply_button, smooth_save_button]),
    smooth_batch_output,
])

smooth_panel = wrap_panel(
    title="Smoothing Preview and Export (Savitzky-Golay)",
    subtitle="Select any NIR or MIR subfolder as source, set the SG window length (odd, > polyorder) and polynomial order, and export all smoothed spectra. Outputs are written beside the selected source folder.",
    controls=smooth_controls,
    status_widget=smooth_status,
    plot_output=smooth_plot_output,
)

display(smooth_panel)
refresh_smooth_files()

In [ ]:
# === Cell 8: Savitzky-Golay Derivative ===

def _build_deriv_source_options():
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted([p for p in parent.iterdir() if p.is_dir()], key=lambda p: p.name)
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options

deriv_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_deriv_source_options(),
    value=_build_deriv_source_options()[0][1] if _build_deriv_source_options() else None,
    width="500px",
)
deriv_file_dropdown = make_dropdown(description="Preview file", options=[], value=None, width="340px")
deriv_order_widget = make_dropdown(
    "Derivative order",
    options=[("1st derivative", 1), ("2nd derivative", 2)],
    value=DEFAULT_SG_DERIV["deriv_order"],
)
deriv_window_widget = make_int("SG window_length", DEFAULT_SG_DERIV["window_length"])
deriv_poly_widget = make_int("SG polyorder", DEFAULT_SG_DERIV["polyorder"])
deriv_random_button = make_button("Random", kind="neutral", icon="random")
deriv_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
deriv_apply_button = make_button("Apply to all files", kind="success", icon="play")
deriv_status = status_html("Select a source folder to preview the Savitzky-Golay derivative.")
deriv_plot_output = make_output(height="430px")
deriv_batch_output = make_output(height="220px")
deriv_save_button = make_save_button()
deriv_figure_holder = FigureHolder("derivative")


def refresh_deriv_files(_=None):
    folder = deriv_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(deriv_file_dropdown, files, keep_value=False)
    preview_derivative()


def preview_derivative(_=None):
    deriv_plot_output.clear_output(wait=True)
    folder = deriv_source_dropdown.value
    filename = deriv_file_dropdown.value

    if folder is None:
        deriv_status.value = '<div class="ir-status">No source folder selected.</div>'
        return
    if filename is None:
        deriv_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    path = Path(folder) / filename
    order = int(deriv_order_widget.value)
    order_label = {1: "1st", 2: "2nd"}.get(order, f"{order}th")

    try:
        input_df = read_spectrum_file(path)
        deriv_df, params = derivative_spectrum(
            input_df,
            window_length=deriv_window_widget.value,
            polyorder=deriv_poly_widget.value,
            deriv_order=order,
        )

        with deriv_plot_output:
            clear_output(wait=True)
            # Input and derivative differ in scale; the derivative uses a
            # secondary y-axis.
            fig, ax = plt.subplots(figsize=(10, 4.8))
            ax.plot(
                input_df["wavenumber"], input_df["signal"],
                label="Input spectrum", **INPUT_STYLE,
            )
            ax.set_xlabel("Wavenumber (1/cm)")
            ax.set_ylabel("Signal")

            ax2 = ax.twinx()
            ax2.plot(
                deriv_df["wavenumber"], deriv_df["signal"],
                label=f"{order_label} derivative", **OUTPUT_STYLE,
            )
            ax2.set_ylabel(f"{order_label} derivative")
            ax2.grid(False)

            if DEFAULT_INVERT_X:
                ax.invert_xaxis()

            ax.set_title(f"Savitzky-Golay {order_label} derivative: {filename}")

            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, loc="best")
            deriv_figure_holder.set(fig, filename)
            plt.show()

        deriv_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b>. '
            f'{order_label} derivative, window_length={params["window_length"]}, '
            f'polyorder={params["polyorder"]}, delta={params["delta"]:.4f} 1/cm.</div>'
        )
    except Exception as exc:
        deriv_status.value = f'<div class="ir-status">Derivative preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_deriv_file(_):
    if not deriv_file_dropdown.options:
        return
    deriv_file_dropdown.value = choose_random_file(deriv_file_dropdown.options)


def save_deriv_figure(_):
    save_panel_figure(deriv_figure_holder, deriv_status)


def refresh_deriv_folder_list(_=None):
    deriv_source_dropdown.options = _build_deriv_source_options()
    refresh_deriv_files()


def apply_derivative_to_all(_):
    deriv_batch_output.clear_output(wait=True)
    folder = deriv_source_dropdown.value
    if folder is None:
        deriv_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        deriv_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    window_length = deriv_window_widget.value
    polyorder = deriv_poly_widget.value
    order = int(deriv_order_widget.value)
    output_parent = Path(folder).parent  # export beside the source folder
    output_prefix = f"deriv{order}_spectra"

    def processor_fn(source_path):
        input_df = read_spectrum_file(source_path)
        deriv_df, params = derivative_spectrum(
            input_df,
            window_length=window_length,
            polyorder=polyorder,
            deriv_order=order,
        )
        return deriv_df, {"status": "success", "n_points": len(deriv_df), **params}

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Derivative",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix=output_prefix,
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: derivative",
                f"timestamp: {datetime.now().isoformat()}",
                f"source_folder: {folder}",
                "method: savitzky-golay",
                f"deriv_order: {order}",
                f"window_length: {window_length}",
                f"polyorder: {polyorder}",
                f"total_attempted: {len(source_files)}",
            ],
        )

        success_count = int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0
        deriv_status.value = (
            f'<div class="ir-status">Derivative batch export completed (order {order}). '
            f'Successes: {success_count}, failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with deriv_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        deriv_status.value = f'<div class="ir-status">Derivative batch export failed: {exc}</div>'


deriv_source_dropdown.observe(refresh_deriv_files, names="value")
deriv_file_dropdown.observe(preview_derivative, names="value")
deriv_order_widget.observe(preview_derivative, names="value")
deriv_window_widget.observe(preview_derivative, names="value")
deriv_poly_widget.observe(preview_derivative, names="value")
deriv_random_button.on_click(choose_random_deriv_file)
deriv_refresh_button.on_click(refresh_deriv_folder_list)
deriv_apply_button.on_click(apply_derivative_to_all)
deriv_save_button.on_click(save_deriv_figure)

deriv_controls = widgets.VBox([
    widgets.HBox([deriv_source_dropdown, deriv_refresh_button]),
    widgets.HBox([deriv_file_dropdown, deriv_random_button]),
    widgets.HBox([deriv_order_widget, deriv_window_widget, deriv_poly_widget, deriv_apply_button, deriv_save_button]),
    deriv_batch_output,
])

deriv_panel = wrap_panel(
    title="Derivative Preview and Export (Savitzky-Golay)",
    subtitle="Select any NIR or MIR subfolder as source, pick the derivative order (1st or 2nd) and SG window/polyorder, and export all derivative spectra. polyorder must be >= derivative order. Outputs are written beside the selected source folder.",
    controls=deriv_controls,
    status_widget=deriv_status,
    plot_output=deriv_plot_output,
)

display(deriv_panel)
refresh_deriv_files()

In [ ]:
# === Cell 9: Unit Conversion ===
# Converts the x-axis between wavelength (nm) and wavenumber (1/cm) via
# nm = 1e7 / cm⁻¹; the signal is unchanged and output is sorted by the new x-axis.


def convert_spectrum_unit(df, direction):
    """Convert the x-axis unit of a spectrum DataFrame.

    Parameters
    ----------
    df : DataFrame
        Must have columns ["wavenumber", "signal"].
    direction : str
        "nm_to_cm" or "cm_to_nm".

    Returns
    -------
    converted : DataFrame
        With columns ["wavenumber", "signal"], sorted ascending by wavenumber.
    params : dict
        Conversion metadata.
    """
    x = df["wavenumber"].to_numpy(dtype=float)
    y = df["signal"].to_numpy(dtype=float)

    if np.any(x == 0):
        raise ValueError("x-axis contains zero — cannot convert (division by zero).")
    if np.any(x < 0):
        raise ValueError("x-axis contains negative values — invalid for unit conversion.")

    x_converted = 1e7 / x

    if direction == "nm_to_cm":
        src_unit, dst_unit = "nm", "1/cm"
    elif direction == "cm_to_nm":
        src_unit, dst_unit = "1/cm", "nm"
    else:
        raise ValueError(f"Unknown direction: {direction!r}. Use 'nm_to_cm' or 'cm_to_nm'.")

    converted = pd.DataFrame({"wavenumber": x_converted, "signal": y})
    converted = converted.sort_values("wavenumber").reset_index(drop=True)

    params = {
        "direction": direction,
        "source_unit": src_unit,
        "target_unit": dst_unit,
        "source_range": f"{float(x.min()):.4f} – {float(x.max()):.4f} {src_unit}",
        "target_range": f"{float(x_converted.min()):.4f} – {float(x_converted.max()):.4f} {dst_unit}",
        "n_points": len(converted),
        "status": "success",
    }
    return converted, params


# --- Widgets ---
def _build_unit_source_options():
    """List all subfolders of both NIR and MIR dataset parents."""
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted([p for p in parent.iterdir() if p.is_dir()], key=lambda p: p.name)
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options


unit_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_unit_source_options(),
    value=_build_unit_source_options()[0][1] if _build_unit_source_options() else None,
    width="500px",
)
unit_file_dropdown = make_dropdown(
    description="Preview file",
    options=[],
    value=None,
    width="340px",
)
unit_direction_toggle = widgets.ToggleButtons(
    options=[("nm → 1/cm", "nm_to_cm"), ("1/cm → nm", "cm_to_nm")],
    value="nm_to_cm",
    description="Direction",
    style=WIDGET_STYLE,
)
unit_random_button = make_button("Random", kind="neutral", icon="random")
unit_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
unit_apply_button = make_button("Apply to all files", kind="success", icon="play")
unit_status = status_html("Select a source folder to preview and export unit-converted spectra.")
unit_plot_output = make_output(height="430px")
unit_batch_output = make_output(height="220px")
unit_save_button = make_save_button()
unit_figure_holder = FigureHolder("unit_conversion")


def refresh_unit_files(_=None):
    folder = unit_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(unit_file_dropdown, files, keep_value=False)
    preview_unit_conversion()


def preview_unit_conversion(_=None):
    unit_plot_output.clear_output(wait=True)
    folder = unit_source_dropdown.value
    filename = unit_file_dropdown.value

    if folder is None:
        unit_status.value = '<div class="ir-status">No source folder selected.</div>'
        return
    if filename is None:
        unit_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    direction = unit_direction_toggle.value
    dir_label = "nm → 1/cm" if direction == "nm_to_cm" else "1/cm → nm"
    src_label = "nm" if direction == "nm_to_cm" else "1/cm"
    dst_label = "1/cm" if direction == "nm_to_cm" else "nm"

    try:
        path = Path(folder) / filename
        input_df = read_spectrum_file(path)
        converted_df, params = convert_spectrum_unit(input_df, direction)

        with unit_plot_output:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.8))

            ax1.plot(
                input_df["wavenumber"], input_df["signal"],
                label=f"Source ({src_label})", **INPUT_STYLE,
            )
            ax1.set_xlabel(f"x-axis ({src_label})")
            ax1.set_ylabel("Signal")
            ax1.set_title(f"Source: {filename}")
            ax1.legend(loc="best")

            ax2.plot(
                converted_df["wavenumber"], converted_df["signal"],
                label=f"Converted ({dst_label})", **OUTPUT_STYLE,
            )
            ax2.set_xlabel(f"x-axis ({dst_label})")
            ax2.set_ylabel("Signal")
            ax2.set_title(f"Converted ({dir_label})")
            ax2.legend(loc="best")

            if DEFAULT_INVERT_X and direction == "nm_to_cm":
                ax2.invert_xaxis()
            elif DEFAULT_INVERT_X and direction == "cm_to_nm":
                ax1.invert_xaxis()

            fig.suptitle(f"Unit conversion: {filename}", fontsize=12, y=1.02)
            fig.tight_layout()
            unit_figure_holder.set(fig, filename)
            plt.show()

        unit_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b>. '
            f'Direction: {dir_label}, '
            f'source: {params["source_range"]}, '
            f'target: {params["target_range"]}, '
            f'{params["n_points"]} points.</div>'
        )
    except Exception as exc:
        unit_status.value = f'<div class="ir-status">Conversion preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_unit_file(_):
    if not unit_file_dropdown.options:
        return
    unit_file_dropdown.value = choose_random_file(unit_file_dropdown.options)


def save_unit_figure(_):
    save_panel_figure(unit_figure_holder, unit_status)


def refresh_unit_folder_list(_=None):
    unit_source_dropdown.options = _build_unit_source_options()
    refresh_unit_files()


def apply_unit_conversion_to_all(_):
    unit_batch_output.clear_output(wait=True)
    folder = unit_source_dropdown.value
    if folder is None:
        unit_status.value = '<div class="ir-status">No source folder selected.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        unit_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    direction = unit_direction_toggle.value
    dir_label = "nm → 1/cm" if direction == "nm_to_cm" else "1/cm → nm"
    output_parent = Path(folder).parent  # export beside the source folder
    output_prefix = "unit_converted_spectra"

    def processor_fn(source_path):
        input_df = read_spectrum_file(source_path)
        converted_df, params = convert_spectrum_unit(input_df, direction)
        return converted_df, params

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Unit Conversion",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix=output_prefix,
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: unit_conversion",
                f"timestamp: {datetime.now().isoformat()}",
                f"source_folder: {folder}",
                f"direction: {dir_label}",
                f"total_attempted: {len(source_files)}",
            ],
        )

        success_count = int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0
        unit_status.value = (
            f'<div class="ir-status">Unit conversion batch export completed ({dir_label}). '
            f'Successes: {success_count}, failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with unit_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        unit_status.value = f'<div class="ir-status">Unit conversion batch export failed: {exc}</div>'


unit_source_dropdown.observe(refresh_unit_files, names="value")
unit_file_dropdown.observe(preview_unit_conversion, names="value")
unit_direction_toggle.observe(preview_unit_conversion, names="value")
unit_random_button.on_click(choose_random_unit_file)
unit_refresh_button.on_click(refresh_unit_folder_list)
unit_apply_button.on_click(apply_unit_conversion_to_all)
unit_save_button.on_click(save_unit_figure)

unit_controls = widgets.VBox([
    widgets.HBox([unit_source_dropdown, unit_refresh_button]),
    widgets.HBox([unit_file_dropdown, unit_random_button]),
    widgets.HBox([unit_direction_toggle, unit_apply_button, unit_save_button]),
    unit_batch_output,
])

unit_panel = wrap_panel(
    title="Unit Conversion Preview and Export (nm ↔ 1/cm)",
    subtitle="Convert the x-axis between wavelength (nm) and wavenumber (1/cm). Both directions use the identity 1e7/x. The signal (y-axis) is unchanged; the output is sorted ascending by the new x-axis. Outputs are written beside the selected source folder.",
    controls=unit_controls,
    status_widget=unit_status,
    plot_output=unit_plot_output,
)

display(unit_panel)
refresh_unit_files()

In [ ]:
# === Cell 10: Axis Resampling ===
# Linearly interpolates spectra onto the wavenumber axis of a reference file
# (e.g. one file from C2_als_z-score/), so the output matches the training data's
# point count and x-positions; typically used after unit conversion or cut.


def resample_spectrum(df, ref_axis):
    """Linearly interpolate a spectrum onto a reference wavenumber axis.

    Parameters
    ----------
    df : DataFrame
        Must have columns ["wavenumber", "signal"].
    ref_axis : ndarray
        Target wavenumber positions (must be sorted ascending).

    Returns
    -------
    resampled : DataFrame
        With columns ["wavenumber", "signal"], length == len(ref_axis).
    params : dict
        Resampling metadata.
    """
    x = df["wavenumber"].to_numpy(dtype=float)
    y = df["signal"].to_numpy(dtype=float)

    # np.interp requires an ascending x-axis
    sort_idx = np.argsort(x)
    x = x[sort_idx]
    y = y[sort_idx]

    # ref_axis must lie within [x.min, x.max]
    x_min, x_max = float(x.min()), float(x.max())
    ref_min, ref_max = float(ref_axis.min()), float(ref_axis.max())

    if ref_min < x_min or ref_max > x_max:
        raise ValueError(
            f"Reference axis [{ref_min:.2f}, {ref_max:.2f}] extends beyond "
            f"source range [{x_min:.2f}, {x_max:.2f}]. "
            f"Cut or convert the source data first so it fully covers the "
            f"reference axis."
        )

    y_interp = np.interp(ref_axis, x, y)

    resampled = pd.DataFrame({"wavenumber": ref_axis, "signal": y_interp})
    params = {
        "status": "success",
        "source_points": len(df),
        "output_points": len(ref_axis),
        "source_range": f"{x_min:.4f} – {x_max:.4f}",
        "ref_range": f"{ref_min:.4f} – {ref_max:.4f}",
    }
    return resampled, params


# --- Reference axis loader ---
_resample_ref_axis = None       # populated by the file picker callback
_resample_ref_label = "none"


def _load_reference_axis(path):
    """Read a spectrum file and extract its wavenumber axis (sorted ascending)."""
    global _resample_ref_axis, _resample_ref_label
    ref_df = read_spectrum_file(path)
    axis = np.sort(ref_df["wavenumber"].to_numpy(dtype=float))
    _resample_ref_axis = axis
    _resample_ref_label = Path(path).name
    return axis


# --- Widgets ---
def _build_resample_source_options():
    """List all subfolders of both NIR and MIR dataset parents."""
    options = []
    for parent in [NIR_DATASET_PARENT, MIR_DATASET_PARENT]:
        if not parent.exists():
            continue
        parent_label = parent.name
        subfolders = sorted(
            [p for p in parent.iterdir() if p.is_dir()],
            key=lambda p: p.name,
        )
        for sf in subfolders:
            options.append((f"{parent_label}/{sf.name}", str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options


resample_source_dropdown = make_dropdown(
    description="Source folder",
    options=_build_resample_source_options(),
    value=_build_resample_source_options()[0][1] if _build_resample_source_options() else None,
    width="500px",
)
resample_file_dropdown = make_dropdown(
    description="Preview file",
    options=[],
    value=None,
    width="340px",
)
resample_ref_text = make_text(
    description="Reference file",
    value="",
    placeholder="Paste path to any training-axis spectrum",
    width="500px",
)
resample_ref_load_button = make_button("Load ref axis", kind="primary", icon="check")
resample_random_button = make_button("Random", kind="neutral", icon="random")
resample_refresh_button = make_button("Refresh folders", kind="primary", icon="refresh")
resample_apply_button = make_button("Apply to all files", kind="success", icon="play")
resample_status = status_html("Paste the path to a reference spectrum, load it, then preview and export.")
resample_plot_output = make_output(height="430px")
resample_batch_output = make_output(height="220px")
resample_save_button = make_save_button()
resample_figure_holder = FigureHolder("resample")


def load_ref_axis_clicked(_):
    ref_path = resample_ref_text.value.strip()
    if not ref_path:
        resample_status.value = '<div class="ir-status">Enter the path to a reference spectrum file first.</div>'
        return
    ref_path = Path(ref_path)
    if not ref_path.is_file():
        resample_status.value = f'<div class="ir-status">Reference file not found: {ref_path}</div>'
        return
    try:
        axis = _load_reference_axis(ref_path)
        resample_status.value = (
            f'<div class="ir-status">Reference axis loaded from <b>{ref_path.name}</b>: '
            f'{len(axis)} points, [{axis.min():.2f}, {axis.max():.2f}].</div>'
        )
        preview_resample()
    except Exception as exc:
        resample_status.value = f'<div class="ir-status">Failed to load reference axis: {exc}</div>'


def refresh_resample_files(_=None):
    folder = resample_source_dropdown.value
    files = discover_spectrum_files(folder) if folder else []
    update_dropdown_from_files(resample_file_dropdown, files, keep_value=False)
    preview_resample()


def preview_resample(_=None):
    resample_plot_output.clear_output(wait=True)
    folder = resample_source_dropdown.value
    filename = resample_file_dropdown.value

    if folder is None:
        resample_status.value = '<div class="ir-status">No source folder selected.</div>'
        return
    if filename is None:
        resample_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return
    if _resample_ref_axis is None:
        resample_status.value = '<div class="ir-status">Load a reference axis first (paste path and click Load).</div>'
        return

    try:
        path = Path(folder) / filename
        input_df = read_spectrum_file(path)
        resampled_df, params = resample_spectrum(input_df, _resample_ref_axis)

        with resample_plot_output:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.8))

            ax1.plot(
                input_df["wavenumber"], input_df["signal"],
                label=f"Source ({params['source_points']} pts)", **INPUT_STYLE,
            )
            ax1.set_xlabel("Wavenumber (1/cm)")
            ax1.set_ylabel("Signal")
            ax1.set_title(f"Source: {filename}")
            ax1.legend(loc="best")
            if DEFAULT_INVERT_X:
                ax1.invert_xaxis()

            ax2.plot(
                resampled_df["wavenumber"], resampled_df["signal"],
                label=f"Resampled ({params['output_points']} pts)", **OUTPUT_STYLE,
            )
            ax2.set_xlabel("Wavenumber (1/cm)")
            ax2.set_ylabel("Signal")
            ax2.set_title(f"Resampled to {_resample_ref_label}")
            ax2.legend(loc="best")
            if DEFAULT_INVERT_X:
                ax2.invert_xaxis()

            fig.suptitle(f"Axis resampling: {filename}", fontsize=12, y=1.02)
            fig.tight_layout()
            resample_figure_holder.set(fig, filename)
            plt.show()

        resample_status.value = (
            f'<div class="ir-status">Preview updated for <b>{filename}</b>. '
            f'Source: {params["source_points"]} pts, '
            f'resampled: {params["output_points"]} pts, '
            f'ref: {_resample_ref_label}.</div>'
        )
    except Exception as exc:
        resample_status.value = f'<div class="ir-status">Resample preview failed for <b>{filename}</b>: {exc}</div>'


def choose_random_resample_file(_):
    if not resample_file_dropdown.options:
        return
    resample_file_dropdown.value = choose_random_file(resample_file_dropdown.options)


def save_resample_figure(_):
    save_panel_figure(resample_figure_holder, resample_status)


def refresh_resample_folder_list(_=None):
    resample_source_dropdown.options = _build_resample_source_options()
    refresh_resample_files()


def apply_resample_to_all(_):
    resample_batch_output.clear_output(wait=True)
    folder = resample_source_dropdown.value
    if folder is None:
        resample_status.value = '<div class="ir-status">No source folder selected.</div>'
        return
    if _resample_ref_axis is None:
        resample_status.value = '<div class="ir-status">Load a reference axis first.</div>'
        return

    source_files = discover_spectrum_files(folder)
    if not source_files:
        resample_status.value = '<div class="ir-status">No files found in the selected source folder.</div>'
        return

    ref_axis = _resample_ref_axis
    output_parent = Path(folder).parent  # export beside the source folder
    output_prefix = "resampled_spectra"

    def processor_fn(source_path):
        input_df = read_spectrum_file(source_path)
        resampled_df, params = resample_spectrum(input_df, ref_axis)
        return resampled_df, params

    try:
        output_dir, log_path, summary_df, failures = export_batch_results(
            step_name="Axis Resampling",
            source_files=source_files,
            processor_fn=processor_fn,
            output_prefix=output_prefix,
            parent_dir=output_parent,
            parameter_lines=[
                "processing_step: axis_resampling",
                f"timestamp: {datetime.now().isoformat()}",
                f"source_folder: {folder}",
                f"reference_file: {_resample_ref_label}",
                f"reference_points: {len(ref_axis)}",
                f"reference_range: [{ref_axis.min():.4f}, {ref_axis.max():.4f}]",
                f"interpolation: linear (np.interp)",
                f"total_attempted: {len(source_files)}",
            ],
        )

        success_count = int((summary_df["status"] == "success").sum()) if not summary_df.empty else 0
        resample_status.value = (
            f'<div class="ir-status">Resampling batch export completed. '
            f'Successes: {success_count}, failures: {len(failures)}. '
            f'Output: {output_dir.name} → {output_parent.name}</div>'
        )

        with resample_batch_output:
            clear_output(wait=True)
            display(summary_df)
            print(f"Parameter log: {log_path}")

    except Exception as exc:
        resample_status.value = f'<div class="ir-status">Resampling batch export failed: {exc}</div>'


resample_source_dropdown.observe(refresh_resample_files, names="value")
resample_file_dropdown.observe(preview_resample, names="value")
resample_ref_load_button.on_click(load_ref_axis_clicked)
resample_random_button.on_click(choose_random_resample_file)
resample_refresh_button.on_click(refresh_resample_folder_list)
resample_apply_button.on_click(apply_resample_to_all)
resample_save_button.on_click(save_resample_figure)

resample_controls = widgets.VBox([
    widgets.HBox([resample_ref_text, resample_ref_load_button]),
    widgets.HBox([resample_source_dropdown, resample_refresh_button]),
    widgets.HBox([resample_file_dropdown, resample_random_button]),
    widgets.HBox([resample_apply_button, resample_save_button]),
    resample_batch_output,
])

resample_panel = wrap_panel(
    title="Axis Resampling Preview and Export",
    subtitle="Interpolate spectra onto a reference wavenumber axis. Paste the path to any spectrum file that already has the target axis (e.g. from the training pipeline), load it, then preview and batch-export. Uses linear interpolation (np.interp). The source data must fully cover the reference axis range.",
    controls=resample_controls,
    status_widget=resample_status,
    plot_output=resample_plot_output,
)

display(resample_panel)
refresh_resample_files()